### Tockeization

Tokenization = splitting text into words




**This decision affects:**

* model performance
* training cost
* multilingual capability
* reasoning ability
* context length
* efficiency

**Why Do We Need Tokenization?**

Computers cannot directly process: "The cat sat on the mat"

Transformers require integers.

We need a mapping: Text
↓
Tokens
↓
Token IDs
↓
Embeddings



**Example:**

"The cat sat" becomes  --> ["The", "cat", "sat"] --> becomes --> [101, 205, 876]



**Why Not Character-Level Tokenization?**

Example: cat -- > becomes --> [c, a, t]

Advantages:
tiny vocabulary ,  handles every word

Problems:
* sequences become very long
* training becomes expensive
* harder to learn semantics


**Example:**

internationalization --> 21 characters

One tokenization decision becomes: 21 separate tokens

Attention cost grows quadratically.


**Subword Tokenization (Modern Solution)**

Modern LLMs use: Subword Tokenization

**Idea:** Split words into reusable pieces. Example:playing   --> becomes: play,ing

**Example:** unhappiness   --> becomes: un,happi,ness

**Benefits:**
* manageable vocabulary
* shorter sequences than characters
* This is why almost every modern LLM uses subword tokenization.

### Types of Teockenization 

**The 3 Main Subword Algorithms:**

**BPE (Byte-Pair Encoding)**: Starts small and merges the most frequently occurring pairs of  characters. (Used by GPT, Llama, Mistral).

**WordPiece:** Starts small and merges pairs that maximize the probability of the training data Uses to mark subword continuations. (Used by BERT)

**Unigram:** Starts with a massive vocabulary and iteratively removes the least useful tokens until it reaches the target size. (Used by T5, ALBERT, Gemini).


### The Software Libraries  

**tiktoken:** OpenAI’s highly optimized library (used for GPT models).

**SentencePiece:** Google’s go-to library that handles both BPE and Unigram (used for Gemini, T5, Llama).

**Proprietary/Custom:** Models like Claude and DeepSeek use closed-source or custom BPE implementations, often approximated by community wrappers.

### Byte Pair Encoding (BPE)  


One of the most important tokenization algorithms.

Used by: GPT-2, GPT-3 , many LLMs


#### Why Was BPE Created?

Suppose your corpus contains:

* cat
* cats
* dog
* dogs
* play
* playing
* played
* player


If we use word-level tokenization, vocabulary becomes:

* cat
* cats
* dog
* dogs
* play
* playing
* played
* player
* ...

Eventually millions of words appear.

Problems:
* huge vocabulary
* memory expensive
* unseen words fail

##Suppose corpus is: Core Idea of BPE

BPE starts with: Characters

Then repeatedly merges the most frequent adjacent pair.

Think: c a t

becomes

c a t --> ca t  ---> then cat

The vocabulary grows gradually.



### Step 1: Training Corpus

Suppose corpus is:

* cat
* cat
* cats
* cats
* dog
* dog
* dogs

**Initially every word is split into characters.**

c a t

c a t

c a t s

c a t s

d o g

d o g

d o g s



Initial vocabulary:

c

a

t

s

d

o

g


**Vocabulary size:**

7



##  2: Count Adjacent Pairs

Look at every neighboring pair.

**Corpus:**

c a t

c a t

c a t s

c a t s




**Pairs:**

(c,a)

(a,t)

(t,s)


**Count frequencies.**

Below is the frequency distribution of the adjacent character pairs in our corpus before applying the BPE merges:

| Pair  | Frequency |
| :---  | :-------- |
| `c a` | 4 |
| `a t` | 4 |
| `t s` | 2 |
| `d o` | 3 |
| `o g` | 3 |
| `g s` | 1 |

Most frequent: c a  and frequency: 4

## Step 3: Merge Most Frequent Pair

Merge:

c a

into:

ca


**Corpus becomes:**

ca t

ca t

ca t s

ca t s

d o g

d o g

d o g s


**Vocabulary becomes:**

c

a

t

s

d

o

g

ca

**Vocabulary size: 8** Note ca is added as extra 

## Step 4: Repeat

Recompute frequencies.

Now pairs:

(ca,t)

(t,s)

(d,o)

(o,g)



#### Step 5: Identify the Next Merge
We evaluate the updated frequencies to find the next pair to merge.

| Pair | Frequency |
| :--- | :-------- |
| `ca t` | 4 |
| `d o`  | 3 |
| `o g`  | 3 |
| `t s`  | 2 |

**Most Frequent Pair:** `ca t` (Count: 4)

> **Action:** The algorithm selects `ca t` and merges it into the new token **`cat`**. This demonstrates how BPE progressively builds complete, meaningful words from smaller character chunks.



## Step 5: Continuing the BPE Merges
After successfully forming the token `cat`, the algorithm evaluates the remaining corpus to find the next most frequent pair.

**Current Corpus State:**
```text
cat
cat
cat s
cat s
d o g
d o g
d o g s



Current Vocabulary: ['c', 'a', 't', 's', 'd', 'o', 'g', 'ca', 'cat']

Action: The most frequent remaining pair is d o (appearing 3 times). The algorithm merges d and o into do, and subsequently merges do and g into dog.



## Step 6: More Merges

**Frequent pairs:**

cat s
dog s



**Merge:**

cats
dogs

**Eventually vocabulary becomes:**

* cat
* cat
* dog
* dogs




## Embedding Matrix


$E \in  R^{Vd}$



where: V = vocabulary size , d = embedding dimension



"cat sat on mat" --> ["cat","sat","on","mat"] -- > [12,45,78,33]


#### Embedding Lookup

We have: [12,45,78,33]

These numbers are only addresses.

For token: 12

Model does: embedding = E[12]

Result:

$X_{cat} = [0.2, 0.8, 0.1, 0.5]$


$X_{sat}​=[0.9,0.3,0.4,0.1]$

$X_{on}​=[0.5,0.2,0.7,0.3]$


$X_{mat}​=[0.6,0.9,0.2,0.4]$

## Input Matrix X

Now stack them together.

$$
X=
\begin{bmatrix}
0.2 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.3 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.7 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.4
\end{bmatrix}
$$



**Shape:**

4×4 Interpretation: 4 tokens, 4-dimensional embedding

### Positional Encoding

Transformer processes everything simultaneously.

Current matrix: X

contains:

cat

sat

on

mat


But it doesn't know:   cat sat on mat  versus   mat on sat cat

So position vectors are added.

**Example:**

Position 1:

$P_1 =[0.1,0,0,0]$ 

Position 2:

$P_2 =[0,0.1,0,0]$

Position 3

$P_3 =[0,0,0.1,0]$

Position 4:

$P_4 =[0,0,0,0.1]$ 

## Positional Encoding

After obtaining the embedding matrix:

$$
X=
\begin{bmatrix}
0.2 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.3 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.7 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.4
\end{bmatrix}
$$

we add positional vectors to encode token order.

Assume:

$$
P=
\begin{bmatrix}
0.1 & 0.0 & 0.0 & 0.0 \\
0.0 & 0.1 & 0.0 & 0.0 \\
0.0 & 0.0 & 0.1 & 0.0 \\
0.0 & 0.0 & 0.0 & 0.1
\end{bmatrix}
$$

The position-aware representation is computed as:

$$
X_{pos}=X+P
$$

For the token cat:

$$
[0.2,0.8,0.1,0.5]
+
[0.1,0.0,0.0,0.0]
=
[0.3,0.8,0.1,0.5]
$$

For the token sat:

$$
[0.9,0.3,0.4,0.1]
+
[0.0,0.1,0.0,0.0]
=
[0.9,0.4,0.4,0.1]
$$

For the token on:

$$
[0.5,0.2,0.7,0.3]
+
[0.0,0.0,0.1,0.0]
=
[0.5,0.2,0.8,0.3]
$$

For the token mat:

$$
[0.6,0.9,0.2,0.4]
+
[0.0,0.0,0.0,0.1]
=
[0.6,0.9,0.2,0.5]
$$

Therefore:

$$
X_{pos}
=
\begin{bmatrix}
0.3 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.4 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.8 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.5
\end{bmatrix}
$$

Shape:

$$
X_{pos} \in \mathbb{R}^{4 \times 4}
$$

At this stage:

- Each row contains semantic information from embeddings.
- Each row also contains positional information.
- The model now knows both token meaning and token order.
- No interaction between tokens has occurred yet.

The resulting matrix becomes the input to the self-attention mechanism.

## Query, Key and Value Generation

Three learnable weight matrices are initialized:

$$
W_Q=
\begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
W_K=
\begin{bmatrix}
1 & 1 \\
0 & 1 \\
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
W_V=
\begin{bmatrix}
1 & 0 \\
1 & 1 \\
0 & 1 \\
1 & 0
\end{bmatrix}
$$

Queries:

$$
Q=X_{pos}W_Q
$$

$$
Q=
\begin{bmatrix}
0.4 & 1.3 \\
1.3 & 0.5 \\
1.3 & 0.5 \\
0.8 & 1.4
\end{bmatrix}
$$

Keys:

$$
K=X_{pos}W_K
$$

$$
K=
\begin{bmatrix}
0.4 & 1.7 \\
1.3 & 1.8 \\
1.3 & 1.0 \\
0.8 & 2.3
\end{bmatrix}
$$

Values:

$$
V=X_{pos}W_V
$$

$$
V=
\begin{bmatrix}
1.6 & 0.9 \\
1.4 & 0.8 \\
1.0 & 1.0 \\
2.0 & 1.1
\end{bmatrix}
$$

## Attention Score Computation

Compute:

$$
QK^T
$$

For the token "sat":

$$
Q_{sat}
=
[1.3,0.5]
$$

Dot products:

$$
score(cat)=
(1.3)(0.4)+(0.5)(1.7)
=
1.37
$$

$$
score(sat)=
(1.3)(1.3)+(0.5)(1.8)
=
2.59
$$

$$
score(on)=
(1.3)(1.3)+(0.5)(1.0)
=
2.19
$$

$$
score(mat)=
(1.3)(0.8)+(0.5)(2.3)
=
2.19
$$

Score vector:

$$
[1.37,2.59,2.19,2.19]
$$

## Scaling

Since:

$$
d_k=2
$$

$$
\sqrt{d_k}=1.414
$$

Scaled scores:

$$
\frac{QK^T}{\sqrt{d_k}}
=
[0.97,1.83,1.55,1.55]
$$

## Softmax

Applying softmax:

$$
Softmax
([0.97,1.83,1.55,1.55])
$$

Produces:

$$
[0.15,0.35,0.25,0.25]
$$

Interpretation:

- 15% attention to cat
- 35% attention to sat
- 25% attention to on
- 25% attention to mat

## Weighted Value Aggregation

Attention output:

$$
Attention(Q,K,V)
=
Softmax(QK^T)V
$$

$$
=
0.15V_{cat}
+
0.35V_{sat}
+
0.25V_{on}
+
0.25V_{mat}
$$

Substituting values:

$$
=
0.15[1.6,0.9]
+
0.35[1.4,0.8]
+
0.25[1.0,1.0]
+
0.25[2.0,1.1]
$$

Result:

$$
[1.48,0.94]
$$

This becomes the new contextual representation for the token "sat".

## Feed Forward Network

The attention output is passed through:

$$
FFN(x)
=
GELU(xW_1+b_1)W_2+b_2
$$

For illustration:

$$
[1.48,0.94]
\rightarrow
[2.31,1.52]
$$

The Feed Forward Network refines and transforms contextual information.

## Output Projection

The final hidden state is projected into vocabulary space:

$$
logits = hW_{vocab}
$$

Example logits:

$$
mat = 8.2
$$

$$
chair = 2.1
$$

$$
floor = 1.4
$$

## Final Softmax

$$
P(token)
=
Softmax(logits)
$$

Result:

$$
mat = 0.84
$$

$$
chair = 0.10
$$

$$
floor = 0.06
$$

## Final Prediction

$$
argmax(P(token))
=
mat
$$

Therefore:

$$
\text{Prediction} = mat
$$

This completes the full Transformer forward pass from embeddings to next-token prediction for the sentence:

$$
\text{"cat sat on mat"}
$$